# 07 - Masked v2 Deployment Training (Fase 2)

Notebook ini menjalankan pelatihan **masked sliding window** untuk artifact deployment v2
(`rain_7d`, `wind_30d`) dari Jupyter Lab, memakai skrip `scripts/train_deployment_v2.py`.

Prasyarat:
- TensorFlow terpasang (`pip install -r requirements.txt`).
- Tabel fitur hasil notebook 01 di-upload manual ke `data/processed/yogyakarta_weather_features.csv`.

Perbedaan v2 vs v1: hari/nilai hilang ditandai `mask=0` dan **tidak** dihitung di
reconstruction error, sehingga gap pendek tidak memaksa menunggu window penuh.


## 1. Konfigurasi


In [ ]:
from pathlib import Path

SERVICE_REPO = Path("..").resolve()
PROCESSED_CSV = SERVICE_REPO / "data" / "processed" / "yogyakarta_weather_features.csv"
OUT_DIR = SERVICE_REPO / "artifacts" / "deployment_v2"
INSTALL_DEPS = False

print("service repo :", SERVICE_REPO)
print("processed csv:", PROCESSED_CSV)
print("output dir   :", OUT_DIR)


## 2. Cek prasyarat


In [ ]:
import sys

PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)
script = SERVICE_REPO / "scripts" / "train_deployment_v2.py"

assert SERVICE_REPO.joinpath("app", "masking.py").exists(), (
    "app/masking.py tidak ditemukan; jalankan notebook dari folder notebooks/ di repo service."
)
assert script.exists(), f"Skrip tidak ditemukan: {script}"

try:
    import tensorflow as tf
    print("tensorflow:", tf.__version__)
except ModuleNotFoundError:
    print("TensorFlow BELUM terpasang. Set INSTALL_DEPS=True atau jalankan: pip install -r requirements.txt")

if PROCESSED_CSV.exists():
    import pandas as pd
    prepared = pd.read_csv(PROCESSED_CSV, parse_dates=["date"], dtype={"station_id": "string"})
    print("total rows:", len(prepared), "| stations:", sorted(prepared["station_id"].astype(str).unique()))
    display(prepared.head())
else:
    print(f"PROCESSED_CSV belum ada: {PROCESSED_CSV}")
    print("Upload hasil notebook 01 (yogyakarta_weather_features.csv) ke path tersebut.")


## 3. Install dependensi (opsional)


In [ ]:
import subprocess

if INSTALL_DEPS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(SERVICE_REPO / "requirements.txt")],
        check=True,
    )
else:
    print("Lewati instalasi dependensi (INSTALL_DEPS=False).")


## 4. Jalankan pelatihan masked v2


In [ ]:
import subprocess

assert PROCESSED_CSV.exists(), f"Upload dulu file processed ke {PROCESSED_CSV}"

cmd = [
    sys.executable, "scripts/train_deployment_v2.py",
    "--processed", str(PROCESSED_CSV),
    "--out", str(OUT_DIR),
    "--epochs", "80", "--patience", "10",
    "--seed", "42",
    "--min-observed-ratio", "0.7",
    "--impute-max-gap-days", "3",
]
print("RUN:", " ".join(cmd), "\n")

process = subprocess.Popen(
    cmd, cwd=str(SERVICE_REPO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in process.stdout:
    print(line, end="")
process.wait()
print("\nexit code:", process.returncode)
assert process.returncode == 0, "Training gagal; cek log di atas."


## 5. Ringkasan hasil training


In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(OUT_DIR / "deployment_model_export_summary.csv")
display(summary)


## 6. Threshold total dan per fitur (v2)


In [ ]:
import json
import pandas as pd
from IPython.display import display

for model in ["rain_7d", "wind_30d"]:
    payload = json.loads((OUT_DIR / model / "threshold.json").read_text(encoding="utf-8"))
    print("=" * 70)
    print(model, "| total thresholds:", payload["total"])
    display(pd.DataFrame(payload["per_feature"]).T)


## 7. Perbandingan threshold v1 vs v2


In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for model in ["rain_7d", "wind_30d"]:
    v2 = json.loads((OUT_DIR / model / "threshold.json").read_text(encoding="utf-8"))["total"]
    v1_path = SERVICE_REPO / "artifacts" / "deployment" / model / "threshold.json"
    v1 = {}
    if v1_path.exists():
        raw = json.loads(v1_path.read_text(encoding="utf-8"))
        v1 = raw.get("total", raw)
    for level in ["p95", "p99", "p995"]:
        rows.append(
            {
                "model": model,
                "level": level,
                "v1": v1.get(level),
                "v2": v2.get(level),
                "delta": None if (level not in v1) else v2[level] - v1[level],
            }
        )
display(pd.DataFrame(rows))


## 8. Uji cepat: service bisa memuat artifact v2


In [ ]:
import sys
sys.path.insert(0, str(SERVICE_REPO))

import pandas as pd
from app.model_runner import LSTMAutoencoderRunner

features = pd.read_csv(PROCESSED_CSV, parse_dates=["date"], dtype={"station_id": "string"})
station = sorted(features["station_id"].astype(str).unique())[0]
station_df = features[features["station_id"].astype(str) == station].sort_values("date").reset_index(drop=True)
print("station:", station, "| rows:", len(station_df))

for model in ["rain_7d", "wind_30d"]:
    runner = LSTMAutoencoderRunner(OUT_DIR / model)
    result = runner.score_latest(station_df)
    print("=" * 70)
    print(model, "->", result.status, "| score:", result.score, "| missing_ratio:", result.missing_ratio)
    print("reason:", result.reason)
    if result.feature_errors:
        print("feature errors:", result.feature_errors)


## 9. Promosi ke produksi

Setelah v2 tervalidasi (bandingkan dengan v1, lalu dry-run di VPS):

```bash
cp -r artifacts/deployment artifacts/deployment_v1_backup
cp -r artifacts/deployment_v2/* artifacts/deployment/
git add artifacts && git commit -m "chore: promote masked v2 deployment artifacts" && git push
```

Lalu di VPS: `git pull` -> `docker build` -> jalankan ulang container (dry-run dulu).
Service otomatis memakai jalur masked scoring begitu `feature_config.json` memuat `mask_policy`.
